# Trend Line Break with Confirmation on SPY
## Strategy Brief
This strategy aims to identify trend reversals in the SPY ETF by detecting breaks in established trend lines, confirmed by additional indicators. When a trend line is broken, it signals a potential reversal, and the confirmation step ensures the validity of the signal, reducing false positives. Trades are executed based on these confirmed signals, with the expectation of capturing significant market moves. Historically, this approach can enhance returns by catching new trends early while minimizing risk during choppy markets.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters necessary for our strategy, including the lookback period for trend line calculation and the confirmation indicator settings.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TREND_LOOKBACK = 20 # Number of periods for trend line
CONFIRMATION_PERIOD = 14 # Period for confirmation indicator, e.g., RSI
RSI_OVERBOUGHT = 70
RSI_OVERSOLD = 30

### PHASE 2 - Data Exploration
We will download historical SPY data, compute the trend line and a confirmation indicator (RSI), and visualize these over the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate RSI as confirmation indicator
def calculate_rsi(data, period=14):
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

data['RSI'] = calculate_rsi(data, CONFIRMATION_PERIOD)

# Plot closing price and RSI
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['RSI'], label='RSI', linestyle='--')
plt.axhline(y=RSI_OVERBOUGHT, color='r', linestyle='-')
plt.axhline(y=RSI_OVERSOLD, color='g', linestyle='-')
plt.title('SPY Price and RSI')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We define the logic for detecting trend line breaks and confirming them with RSI. The signal series will indicate when to enter or exit positions.

In [ ]:
# Detect trend line breaks
highs = data['Close'].rolling(window=TREND_LOOKBACK).max()
lows = data['Close'].rolling(window=TREND_LOOKBACK).min()
data['Trend_Break'] = (data['Close'] > highs.shift(1)) | (data['Close'] < lows.shift(1))

# Confirm with RSI
data['Signal'] = 0
long_condition = (data['Trend_Break'] & (data['RSI'] < RSI_OVERSOLD))
short_condition = (data['Trend_Break'] & (data['RSI'] > RSI_OVERBOUGHT))
data.loc[long_condition, 'Signal'] = 1
data.loc[short_condition, 'Signal'] = -1

# Positions based on signals
data['Position'] = data['Signal'].replace(to_replace=0, method='ffill')

### PHASE 4 - Coding & Backtesting
We will simulate trades by shifting positions to avoid look-ahead bias, calculate daily returns, and plot the equity curve.

In [ ]:
# Shift positions to simulate trading at next day's open
data['Position'] = data['Position'].shift(1)

# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Position'] * data['Market_Return']

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of Trend Line Break Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics like CAGR, Sharpe, Sortino, Calmar ratios, and max drawdown, comparing it to a buy-and-hold strategy.

In [ ]:
# Calculate performance metrics
def calculate_cagr(equity_curve):
    n_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    return (equity_curve[-1] / equity_curve[0]) ** (1 / n_years) - 1

def calculate_sharpe(returns, rf_rate=0.01):
    excess_returns = returns - rf_rate / 252
    return np.sqrt(252) * excess_returns.mean() / excess_returns.std()

def calculate_sortino(returns, rf_rate=0.01):
    excess_returns = returns - rf_rate / 252
    downside_std = returns[returns < 0].std()
    return np.sqrt(252) * excess_returns.mean() / downside_std

def calculate_max_drawdown(equity_curve):
    drawdown = equity_curve / equity_curve.cummax() - 1
    return drawdown.min()

strategy_cagr = calculate_cagr(data['Equity_Curve'])
strategy_sharpe = calculate_sharpe(data['Strategy_Return'])
strategy_sortino = calculate_sortino(data['Strategy_Return'])
strategy_max_drawdown = calculate_max_drawdown(data['Equity_Curve'])

# Buy and hold metrics
data['Buy_Hold_Equity'] = (1 + data['Market_Return']).cumprod()
buy_hold_cagr = calculate_cagr(data['Buy_Hold_Equity'])
buy_hold_max_drawdown = calculate_max_drawdown(data['Buy_Hold_Equity'])

# Comparison table
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_max_drawdown],
    'Buy and Hold': [buy_hold_cagr, np.nan, np.nan, buy_hold_max_drawdown]
})
print(performance_df)

### PHASE 6 - Deploy & Monitor
We will create a function to download recent data, compute today's signal, and determine the current position.

In [ ]:
def get_latest_signal():
    recent_data = yf.download('SPY', start=pd.Timestamp.today() - pd.Timedelta(days=60), end=pd.Timestamp.today())
    recent_data['RSI'] = calculate_rsi(recent_data, CONFIRMATION_PERIOD)
    highs = recent_data['Close'].rolling(window=TREND_LOOKBACK).max()
    lows = recent_data['Close'].rolling(window=TREND_LOOKBACK).min()
    recent_data['Trend_Break'] = (recent_data['Close'] > highs.shift(1)) | (recent_data['Close'] < lows.shift(1))
    
    long_condition = (recent_data['Trend_Break'] & (recent_data['RSI'] < RSI_OVERSOLD))
    short_condition = (recent_data['Trend_Break'] & (recent_data['RSI'] > RSI_OVERBOUGHT))
    
    if long_condition.iloc[-1]:
        print('Current Position: Long')
    elif short_condition.iloc[-1]:
        print('Current Position: Short')
    else:
        print('Current Position: Neutral')

get_latest_signal()